In [ ]:
import sys

sys.path.append("..")
import os
from intelligence_toolkit.detect_entity_networks.api import DetectEntityNetworks
from intelligence_toolkit.AI.openai_configuration import OpenAIConfiguration
import polars as pl

In [ ]:
# Create the workflow object
den = DetectEntityNetworks()
# Set the AI configuration
ai_configuration = OpenAIConfiguration(
    {
        "api_type": "OpenAI",
        "api_key": os.environ["OPENAI_API_KEY"],
        "model": "gpt-4o",
    }
)
den.set_ai_configuration(ai_configuration)

data_path = "../example_outputs/detect_entity_networks/company_grievances/company_grievances_input.csv"
entity_df = pl.read_csv(data_path)

print("Loaded data")

In [ ]:
# set entity-attributes
from intelligence_toolkit.detect_entity_networks.prepare_model import (
    format_data_columns,
)


entity_id_column = "name"
columns_to_link = ["address", "city", "email", "phone", "owner"]
entity_df = format_data_columns(entity_df, columns_to_link, entity_id_column)
den.add_attribute_links(entity_df, entity_id_column, columns_to_link)

summary = den.get_model_summary_value()
print("Summary")
print(summary)

In [ ]:
# set flags
from intelligence_toolkit.detect_entity_networks.classes import FlagAggregatorType


entity_id_column = "name"
columns_to_link = [
    "safety_grievances",
    "pay_grievances",
    "conditions_grievances",
    "treatment_grievances",
    "workload_grievances",
]
flag_format = FlagAggregatorType.Count
den.add_flag_links(entity_df, entity_id_column, columns_to_link, flag_format)
summary = den.get_model_summary_value()
print("Summary")
print(summary)

In [ ]:
# set groups
entity_id_column = "name"
columns_to_link = ["sector", "country"]
den.add_group_links(entity_df, entity_id_column, columns_to_link)

summary = den.get_model_summary_value()
print("Summary")
print(summary)

In [ ]:
await den.index_nodes(["ENTITY"])
if len(den.embedded_texts) > 0:
    print(f"Number of nodes indexed: {len(den.embedded_texts)}")

In [ ]:
# infer nodes with similar names

threshold = 0.03
den.infer_nodes(threshold)

inferred_links_count = len(den.inferred_links)
if inferred_links_count > 0:
    print(f"Number of links inferred: {inferred_links_count}")
    inferred_df = den.inferred_nodes_df()
    print(inferred_df)
else:
    print("No inferred links")

In [ ]:
den.identify()

In [ ]:
print(den.get_records_summary())
print(f"Attributes removed because of high degree: {len(den.trimmed_attributes)}")
print(den.trimmed_attributes)

In [ ]:
# with entities
den.get_entity_df().head()

In [ ]:
# with entities and groups
group_df = den.get_grouped_df()
group_df.head()

In [ ]:
earths = group_df.filter(pl.col("entity_id") == "Earths Bounty").head()
print(earths)

In [ ]:
selected_network = earths[0].get_column("network_id")[0]
print("Selected Network: ", selected_network)

In [ ]:
den.get_exposure_report("Earths Bounty", selected_network)

In [ ]:
# report
selected_entity = "Earths Bounty"
report = den.generate_report(selected_network=selected_network)
print(report)